# Prediccion de abandono de clientes (Churn) — Telco

Una empresa de telecomunicaciones pierde clientes constantemente — el
fenomeno se llama churn. Perder un cliente cuesta mucho mas que retenerlo,
por eso predecir quien esta en riesgo de irse permite actuar a tiempo.

Este notebook explora el dataset de clientes de Telco para entender que
variables se relacionan con el abandono, como paso previo a construir un
modelo de clasificacion.

## Carga de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/churn.csv')

print(f"Dimensiones: {df.shape[0]} filas * {df.shape[1]} columnas")
df.head()

Dimensiones: 7043 filas * 21 columnas


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Diccionario de columnas

- **customerID**: identificador unico, se descarta del modelo
- **Demograficas**: gender, SeniorCitizen, Partner, Dependents
- **Servicio telefonico**: tenure, PhoneService, MultipleLines
- **Servicio de internet**: InternetService y sus add-ons (OnlineSecurity,
  OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies)
- **Contrato y facturacion**: Contract, PaperlessBilling, PaymentMethod,
  MonthlyCharges, TotalCharges
- **Target**: Churn (Yes/No) — variable a predecir

In [3]:
#valores unicos de cada columna categorica
columnas_categoricas = df.select_dtypes(include=['object', 'string']).columns

for col in columnas_categoricas:
    print(f"{col}: {df[col].unique()}")
    print()

customerID: <StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str

gender: <StringArray>
['Female', 'Male']
Length: 2, dtype: str

Partner: <StringArray>
['Yes', 'No']
Length: 2, dtype: str

Dependents: <StringArray>
['No', 'Yes']
Length: 2, dtype: str

PhoneService: <StringArray>
['No', 'Yes']
Length: 2, dtype: str

MultipleLines: <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

InternetService: <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

OnlineSecurity: <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

OnlineBackup: <StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

DeviceProtection: <StringArray>
['No', 'Yes', 'No internet s